## Phase 1

In [1]:
pip install -q drain3 pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import zipfile
import shutil
import urllib.request

import pandas as pd
from tqdm.auto import tqdm

BASE_DIR = Path.cwd()
WORK_DIR = BASE_DIR / "day-2" if (BASE_DIR / "day-2").exists() else BASE_DIR

DATA_DIR = WORK_DIR / "data"
RESULTS_DIR = WORK_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Full HDFS_v1: ~186 MB zip nên muốn máy yếu hoặc test nhanh, đổi DATASET_MODE = "sample".

DATASET_MODE = "sample"

if DATASET_MODE == "full":
    url = "https://zenodo.org/records/8196385/files/HDFS_v1.zip?download=1"
    zip_path = DATA_DIR / "HDFS_v1.zip"
    extract_dir = DATA_DIR / "HDFS_v1"

    if not zip_path.exists():
        print("Downloading HDFS_v1.zip...")
        with urllib.request.urlopen(url) as response, open(zip_path, "wb") as f:
            shutil.copyfileobj(response, f)

    if not extract_dir.exists():
        print("Extracting HDFS_v1.zip...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

    log_candidates = list(extract_dir.rglob("*.log"))
    log_path = max(log_candidates, key=lambda p: p.stat().st_size)

else:
    url = "https://raw.githubusercontent.com/logpai/loghub/master/HDFS/HDFS_2k.log"
    log_path = DATA_DIR / "HDFS_2k.log"

    if not log_path.exists():
        print("Downloading HDFS_2k.log...")
        with urllib.request.urlopen(url) as response, open(log_path, "wb") as f:
            shutil.copyfileobj(response, f)

print("Log file:", log_path)

Log file: d:\DevopsAndCloud\AIOPS\W1\day-2\data\HDFS_2k.log


In [10]:
def count_lines(path: Path) -> int:
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        return sum(1 for _ in f)

total_lines = count_lines(log_path)
print(f"Total log lines: {total_lines:,}")

Total log lines: 2,000


### Hàm parse toàn bộ log bằng Drain3

In [11]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

def iter_log_lines(path: Path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            yield line.rstrip("\r\n")

def get_clusters(miner):
    clusters = miner.drain.clusters
    if isinstance(clusters, dict):
        return list(clusters.values())
    return list(clusters)

def cluster_template(cluster):
    if hasattr(cluster, "get_template"):
        return cluster.get_template()
    return " ".join(cluster.log_template)

def parse_with_drain(log_path: Path, sim_th: float, depth: int = 4):
    config = TemplateMinerConfig()
    config.drain_sim_th = sim_th
    config.drain_depth = depth
    config.profiling_enabled = False

    miner = TemplateMiner(config=config)

    n = 0
    for line in tqdm(iter_log_lines(log_path), total=total_lines, desc=f"Drain3 sim_th={sim_th}"):
        miner.add_log_message(line)
        n += 1

    rows = []
    for cluster in get_clusters(miner):
        rows.append({
            "template_id": cluster.cluster_id,
            "template": cluster_template(cluster),
            "count": cluster.size,
        })

    templates_df = (
        pd.DataFrame(rows)
        .sort_values(["count", "template_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

    return miner, templates_df, n

### Parse với giá trị baseline drain_sim_th=0.5

In [12]:
miner, templates_df, parsed_lines = parse_with_drain(log_path, sim_th=0.5)

print(f"Parsed lines: {parsed_lines:,}")
print(f"Unique templates: {len(templates_df):,}")

pd.set_option("display.max_colwidth", 220)
templates_df

Drain3 sim_th=0.5: 100%|██████████| 2000/2000 [00:00<00:00, 115480.35it/s]

Parsed lines: 2,000
Unique templates: 21


,template_id,template,count
0,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>,314
1,1,<*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating,311
2,3,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>,292
3,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>,292
4,7,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>,263
5,11,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>,224
6,9,<*> <*> <*> WARN dfs.DataNode$DataXceiver: <*> exception while serving <*> to <*>,80
7,12,081110 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>,59
8,16,081111 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,55
9,10,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,52


In [13]:
top10 = templates_df.head(10).copy()

top10_path = RESULTS_DIR / "top_templates.csv"
top10.to_csv(top10_path, index=False)

# Optional: lưu luôn toàn bộ templates để dễ nộp / kiểm tra
all_templates_path = RESULTS_DIR / "all_templates.csv"
templates_df.to_csv(all_templates_path, index=False)

print("Saved:", top10_path)
top10

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\top_templates.csv


,template_id,template,count
0,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>,314
1,1,<*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating,311
2,3,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>,292
3,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>,292
4,7,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>,263
5,11,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>,224
6,9,<*> <*> <*> WARN dfs.DataNode$DataXceiver: <*> exception while serving <*> to <*>,80
7,12,081110 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>,59
8,16,081111 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,55
9,10,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,52


### Tune drain_sim_th với 0.3, 0.5, 0.7

In [14]:
tuning_rows = []

for sim_th in [0.3, 0.5, 0.7]:
    if sim_th == 0.5:
        num_templates = len(templates_df)
    else:
        _, df_tmp, _ = parse_with_drain(log_path, sim_th=sim_th)
        num_templates = len(df_tmp)

    tuning_rows.append({
        "drain_sim_th": sim_th,
        "num_templates": num_templates,
    })

tuning_df = pd.DataFrame(tuning_rows)

tuning_path = RESULTS_DIR / "drain_tuning.csv"
tuning_df.to_csv(tuning_path, index=False)

print("Saved:", tuning_path)
tuning_df

Drain3 sim_th=0.7: 100%|██████████| 2000/2000 [00:00<00:00, 13636.00it/s]


Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\drain_tuning.csv


,drain_sim_th,num_templates
0,0.3,17
1,0.5,21
2,0.7,820


In [15]:
s = tuning_df.set_index("drain_sim_th")["num_templates"]

if s.loc[0.7] <= s.loc[0.5] * 1.2:
    best_sim_th = 0.7
    reason = "0.7 không làm số template tăng mạnh so với 0.5, nên giữ được nhiều chi tiết hơn."
elif s.loc[0.5] <= s.loc[0.3] * 1.8:
    best_sim_th = 0.5
    reason = "0.5 cân bằng giữa gom nhóm và tách chi tiết; 0.3 có thể gom hơi mạnh, 0.7 dễ tách quá nhiều."
else:
    best_sim_th = 0.3
    reason = "0.5/0.7 làm số template tăng mạnh, có dấu hiệu over-splitting; 0.3 ổn định hơn."

print(f"Best drain_sim_th: {best_sim_th}")
print("Reason:", reason)

Best drain_sim_th: 0.5
Reason: 0.5 cân bằng giữa gom nhóm và tách chi tiết; 0.3 có thể gom hơi mạnh, 0.7 dễ tách quá nhiều.


## PHASE 2

### Parse lại log và lưu line-level result


In [29]:
import re
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import IsolationForest

best_sim_th = globals().get("best_sim_th", 0.5)

config_p2 = TemplateMinerConfig()
config_p2.drain_sim_th = best_sim_th
config_p2.drain_depth = 4
config_p2.profiling_enabled = False

miner_p2 = TemplateMiner(config=config_p2)

block_re = re.compile(r"blk_-?\d+")

def parse_hdfs_timestamp(line: str):
    parts = line.split()
    if len(parts) < 2:
        return pd.NaT
    return pd.to_datetime(parts[0] + " " + parts[1], format="%y%m%d %H%M%S", errors="coerce")

parsed_rows = []

for idx, line in enumerate(tqdm(iter_log_lines(log_path), total=total_lines, desc="Parse Phase 2")):
    result = miner_p2.add_log_message(line)
    parsed_rows.append({
        "line_id": idx,
        "timestamp": parse_hdfs_timestamp(line),
        "template_id": result["cluster_id"],
        "block_ids": block_re.findall(line),
        "raw_log": line,
    })

parsed_df = pd.DataFrame(parsed_rows)
parsed_df["timestamp"] = pd.to_datetime(parsed_df["timestamp"])

clusters = get_clusters(miner_p2)

final_templates_df = pd.DataFrame([
    {
        "template_id": cluster.cluster_id,
        "template": cluster_template(cluster),
        "count": cluster.size,
    }
    for cluster in clusters
]).sort_values("template_id").reset_index(drop=True)

parsed_path = RESULTS_DIR / "parsed_logs_phase2.csv"
parsed_df.to_csv(parsed_path, index=False)

templates_path = RESULTS_DIR / "phase2_final_templates.csv"
final_templates_df.to_csv(templates_path, index=False)

print("Parsed rows:", len(parsed_df))
print("Unique template_id:", parsed_df["template_id"].nunique())
print("Final templates:", len(final_templates_df))
print("Time range:", parsed_df["timestamp"].min(), "->", parsed_df["timestamp"].max())

final_templates_df.head()

Parse Phase 2: 100%|██████████| 2000/2000 [00:00<00:00, 8309.72it/s]

Parsed rows: 2000
Unique template_id: 21
Final templates: 21
Time range: 2008-11-09 20:36:15 -> 2008-11-11 10:20:17


,template_id,template,count
0,1,<*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating,311
1,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>,314
2,3,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>,292
3,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>,292
4,5,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,8


## Tạo template count time series

In [30]:
# Với HDFS_2k sample, dùng 1h để tránh quá sparse.
WINDOW = "1h"

template_ts = (
    parsed_df
    .dropna(subset=["timestamp"])
    .groupby([pd.Grouper(key="timestamp", freq=WINDOW), "template_id"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

template_ts_path = RESULTS_DIR / "template_count_timeseries.csv"
template_ts.to_csv(template_ts_path)

print("Saved:", template_ts_path)
print("Shape:", template_ts.shape)
template_ts.head()

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\template_count_timeseries.csv
Shape: (39, 21)


template_id,1,2,3,4,5,6,7,8,9,10,...,12,13,14,15,16,17,18,19,20,21
timestamp,,,,,,,,,,,,,,,,,,,,,
2008-11-09 20:00:00,6,9,6,4,3,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2008-11-09 21:00:00,7,9,18,6,2,1,1,7,7,0,...,0,0,0,0,0,0,0,0,0,0
2008-11-09 22:00:00,0,0,0,0,0,0,0,2,13,0,...,0,0,0,0,0,0,0,0,0,0
2008-11-09 23:00:00,12,10,8,14,3,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2008-11-10 00:00:00,9,6,5,6,0,1,0,0,0,3,...,0,0,0,0,0,0,0,0,0,0


## Detect template spike bằng 3-sigma

In [32]:
def detect_3sigma_spikes(ts_df, min_count=2, z_threshold=3.0):
    events = []

    for template_id in ts_df.columns:
        series = ts_df[template_id]
        mean = series.mean()
        std = series.std(ddof=0)

        if std == 0 or np.isnan(std):
            continue

        z = (series - mean) / std
        spike_mask = (z >= z_threshold) & (series >= min_count)

        for ts, count in series[spike_mask].items():
            events.append({
                "timestamp": ts,
                "template_id": template_id,
                "count": int(count),
                "mean": float(mean),
                "std": float(std),
                "z_score": float(z.loc[ts]),
            })

    return pd.DataFrame(events)

spike_df = detect_3sigma_spikes(template_ts, min_count=2, z_threshold=3.0)

if len(spike_df) > 0:
    spike_df = spike_df.merge(
        final_templates_df[["template_id", "template"]],
        on="template_id",
        how="left"
    )
    spike_df = spike_df.sort_values(["z_score", "count"], ascending=[False, False])
else:
    spike_df = pd.DataFrame(columns=["timestamp", "template_id", "count", "mean", "std", "z_score", "template"])

spike_path = RESULTS_DIR / "template_spikes_3sigma.csv"
spike_df.to_csv(spike_path, index=False)

print("Saved:", spike_path)
print("Detected spike events:", len(spike_df))
spike_df.head(20)

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\template_spikes_3sigma.csv
Detected spike events: 13


,timestamp,template_id,count,mean,std,z_score,template
4,2008-11-09 21:00:00,8,7,0.230769,1.142681,5.923990,081109 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
7,2008-11-10 01:00:00,10,15,1.333333,2.964174,4.610616,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
8,2008-11-10 10:00:00,11,64,5.743590,12.899449,4.516194,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>
9,2008-11-10 08:00:00,12,13,1.512821,2.610352,4.400624,081110 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
10,2008-11-11 00:00:00,15,5,0.307692,1.089668,4.306180,081111 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
3,2008-11-10 10:00:00,7,68,6.743590,15.017456,4.079014,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>
0,2008-11-09 20:00:00,5,3,0.205128,0.722513,3.868265,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
1,2008-11-09 23:00:00,5,3,0.205128,0.722513,3.868265,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
2,2008-11-10 08:00:00,6,3,0.512821,0.674510,3.687387,<*> <*> 13 INFO dfs.DataBlockScanner: Verification succeeded for <*>
11,2008-11-11 02:00:00,15,4,0.307692,1.089668,3.388470,081111 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>


### Detect template mới xuất hiện khi nào

In [33]:
first_seen_df = (
    parsed_df
    .dropna(subset=["timestamp"])
    .groupby("template_id", as_index=False)
    .agg(first_seen=("timestamp", "min"))
    .merge(final_templates_df, on="template_id", how="left")
    .sort_values("first_seen")
)

first_seen_path = RESULTS_DIR / "new_templates_first_seen.csv"
first_seen_df.to_csv(first_seen_path, index=False)

print("Saved:", first_seen_path)
print("Number of templates:", len(first_seen_df))
first_seen_df.head(20)

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\new_templates_first_seen.csv
Number of templates: 21


,template_id,first_seen,template,count
0,1,2008-11-09 20:36:15,<*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating,311
1,2,2008-11-09 20:40:05,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>,314
2,3,2008-11-09 20:46:55,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>,292
3,4,2008-11-09 20:48:15,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>,292
4,5,2008-11-09 20:50:35,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,8
5,6,2008-11-09 20:59:31,<*> <*> 13 INFO dfs.DataBlockScanner: Verification succeeded for <*>,20
6,7,2008-11-09 21:38:37,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>,263
7,8,2008-11-09 21:38:47,081109 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>,9
8,9,2008-11-09 21:40:43,<*> <*> <*> WARN dfs.DataNode$DataXceiver: <*> exception while serving <*> to <*>,80
9,10,2008-11-10 00:01:17,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,52


### Isolation Forest trên template count

In [34]:
X = template_ts.copy()

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
)

window_pred = iso.fit_predict(X)
window_score = iso.decision_function(X)

iforest_windows = pd.DataFrame({
    "timestamp": X.index,
    "iforest_score": window_score,
    "is_anomaly": window_pred == -1,
}).sort_values("iforest_score")

iforest_path = RESULTS_DIR / "iforest_anomaly_windows.csv"
iforest_windows.to_csv(iforest_path, index=False)

print("Saved:", iforest_path)
print("Anomaly windows:", int(iforest_windows["is_anomaly"].sum()))
iforest_windows.head(20)

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\iforest_anomaly_windows.csv
Anomaly windows: 2


,timestamp,iforest_score,is_anomaly
25,2008-11-10 21:00:00,-0.029399,True
34,2008-11-11 06:00:00,-0.018833,True
36,2008-11-11 08:00:00,0.002093,False
12,2008-11-10 08:00:00,0.031999,False
1,2008-11-09 21:00:00,0.037355,False
14,2008-11-10 10:00:00,0.047073,False
37,2008-11-11 09:00:00,0.047797,False
26,2008-11-10 22:00:00,0.051859,False
5,2008-11-10 01:00:00,0.077478,False
35,2008-11-11 07:00:00,0.086778,False


### template nào đóng góp nhiều nhất Với mỗi anomaly window

In [35]:
anomaly_times = iforest_windows.loc[iforest_windows["is_anomaly"], "timestamp"]

top_templates_in_anomaly_windows = []

for ts in anomaly_times:
    row = template_ts.loc[ts]
    top_counts = row[row > 0].sort_values(ascending=False).head(5)

    for template_id, count in top_counts.items():
        template_text = final_templates_df.loc[
            final_templates_df["template_id"] == template_id,
            "template"
        ].iloc[0]

        top_templates_in_anomaly_windows.append({
            "timestamp": ts,
            "template_id": template_id,
            "count": int(count),
            "template": template_text,
        })

iforest_top_df = pd.DataFrame(top_templates_in_anomaly_windows)

iforest_top_path = RESULTS_DIR / "iforest_top_templates_in_anomaly_windows.csv"
iforest_top_df.to_csv(iforest_top_path, index=False)

print("Saved:", iforest_top_path)
iforest_top_df.head(20)

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\iforest_top_templates_in_anomaly_windows.csv


,timestamp,template_id,count,template
0,2008-11-10 21:00:00,7,51,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>
1,2008-11-10 21:00:00,11,36,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>
2,2008-11-10 21:00:00,1,21,<*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating
3,2008-11-10 21:00:00,4,21,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>
4,2008-11-10 21:00:00,3,20,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>
5,2008-11-11 06:00:00,7,22,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>
6,2008-11-11 06:00:00,1,17,<*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating
7,2008-11-11 06:00:00,11,17,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>
8,2008-11-11 06:00:00,2,15,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
9,2008-11-11 06:00:00,3,15,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>


### Tải HDFS label và gán label theo block id

In [36]:
label_path = DATA_DIR / "HDFS_anomaly_label.csv"
hdfs_zip_path = DATA_DIR / "HDFS_v1.zip"

if label_path.exists():
    first_text = label_path.read_text(encoding="utf-8", errors="ignore")[:500].lower()
    if "<!doctype html" in first_text or "<html" in first_text or "blockid,label" not in first_text:
        print("Removing invalid label file:", label_path)
        label_path.unlink()

if not label_path.exists():
    hdfs_url = "https://zenodo.org/records/8196385/files/HDFS_v1.zip?download=1"

    if not hdfs_zip_path.exists():
        print("Downloading HDFS_v1.zip for anomaly labels...")
        with urllib.request.urlopen(hdfs_url) as response, open(hdfs_zip_path, "wb") as f:
            shutil.copyfileobj(response, f)

    print("Extracting anomaly_label.csv from HDFS_v1.zip...")
    with zipfile.ZipFile(hdfs_zip_path, "r") as zf:
        label_members = [
            name for name in zf.namelist()
            if name.lower().endswith("anomaly_label.csv")
        ]

        if not label_members:
            raise FileNotFoundError("Cannot find anomaly_label.csv inside HDFS_v1.zip")

        label_member = label_members[0]
        print("Found:", label_member)

        with zf.open(label_member) as src, open(label_path, "wb") as dst:
            shutil.copyfileobj(src, dst)

labels_df = pd.read_csv(label_path)

print("Label file:", label_path)
print("Columns:", labels_df.columns.tolist())
print(labels_df.head())
print(labels_df["Label"].value_counts())

Label file: d:\DevopsAndCloud\AIOPS\W1\day-2\data\HDFS_anomaly_label.csv
Columns: ['BlockId', 'Label']
                    BlockId    Label
0  blk_-1608999687919862906   Normal
1   blk_7503483334202473044   Normal
2  blk_-3544583377289625738  Anomaly
3  blk_-9073992586687739851   Normal
4   blk_7854771516489510256   Normal
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64


###  Gán ground truth anomaly cho từng log line

In [37]:
block_label_map = dict(zip(labels_df["BlockId"], labels_df["Label"]))

def line_is_anomaly(block_ids):
    return any(block_label_map.get(block_id) == "Anomaly" for block_id in block_ids)

parsed_df["true_anomaly"] = parsed_df["block_ids"].apply(line_is_anomaly)

print(parsed_df["true_anomaly"].value_counts())
parsed_df[["timestamp", "template_id", "true_anomaly", "raw_log"]].head()

true_anomaly
False    1930
True       70
Name: count, dtype: int64


,timestamp,template_id,true_anomaly,raw_log
0,2008-11-09 20:36:15,1,False,081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
1,2008-11-09 20:38:07,1,False,081109 203807 222 INFO dfs.DataNode$PacketResponder: PacketResponder 0 for block blk_-6952295868487656571 terminating
2,2008-11-09 20:40:05,2,False,081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
3,2008-11-09 20:40:15,1,False,081109 204015 308 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_8229193803249955061 terminating
4,2008-11-09 20:41:06,1,False,081109 204106 329 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_-6670958622368987959 terminating


### Evaluate precision/recall ở window-level

In [38]:
def evaluate_detector(y_true, y_pred, name):
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[False, True]).ravel()

    return {
        "detector": name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "false_alarms": int(fp),
    }

parsed_df["window"] = parsed_df["timestamp"].dt.floor(WINDOW)

# Ground truth window: window có ít nhất 1 dòng anomaly thì xem là anomaly window
window_true = (
    parsed_df
    .groupby("window")["true_anomaly"]
    .any()
    .reindex(template_ts.index, fill_value=False)
)

# 3-sigma prediction: window có ít nhất 1 template spike
spike_windows = set(pd.to_datetime(spike_df["timestamp"]))
window_pred_3sigma = pd.Series(
    template_ts.index.isin(spike_windows),
    index=template_ts.index
)

# Isolation Forest prediction: window bị model đánh dấu anomaly
iforest_anomaly_windows = set(pd.to_datetime(anomaly_times))
window_pred_iforest = pd.Series(
    template_ts.index.isin(iforest_anomaly_windows),
    index=template_ts.index
)

metrics_df = pd.DataFrame([
    evaluate_detector(window_true, window_pred_3sigma, "Template Count 3-sigma Window"),
    evaluate_detector(window_true, window_pred_iforest, "Isolation Forest Window"),
])

metrics_path = RESULTS_DIR / "phase2_log_anomaly_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("Saved:", metrics_path)
print("Window ground truth:")
print(window_true.value_counts())
metrics_df

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\phase2_log_anomaly_metrics.csv
Window ground truth:
true_anomaly
True     21
False    18
Name: count, dtype: int64


,detector,precision,recall,f1,tp,fp,fn,tn,false_alarms
0,Template Count 3-sigma Window,0.7,0.333333,0.451613,7,3,14,15,3
1,Isolation Forest Window,1.0,0.095238,0.173913,2,0,19,18,0


Evaluation được tính ở window-level vì detector hoạt động trên template count time series theo window. HDFS label gốc là block-level, nên map BlockId sang log line trước, sau đó một window được xem là anomaly nếu có ít nhất một anomaly log line trong window đó.

Trên HDFS_2k sample, 3-sigma có F1 tốt hơn Isolation Forest vì recall cao hơn. Isolation Forest precision = 1.0 và false alarm = 0, nhưng recall thấp, nghĩa là model quá conservative trên sample nhỏ.

## Phase 3

### TF-IDF embedding trên templates

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

templates_for_embedding = final_templates_df[["template_id", "template", "count"]].copy()

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[\w$.*:/<>-]+\b"
)

tfidf_matrix = vectorizer.fit_transform(templates_for_embedding["template"])

similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=templates_for_embedding["template_id"],
    columns=templates_for_embedding["template_id"],
)

similarity_path = RESULTS_DIR / "template_similarity_matrix.csv"
similarity_df.to_csv(similarity_path)

print("Saved:", similarity_path)
print("TF-IDF matrix shape:", tfidf_matrix.shape)
similarity_df.head()

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\template_similarity_matrix.csv
TF-IDF matrix shape: (21, 274)


template_id,1,2,3,4,5,6,7,8,9,10,...,12,13,14,15,16,17,18,19,20,21
template_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.042498,0.251040,0.057847,0.075338,0.203678,0.057391,0.069073,0.000000,0.080792,...,0.073204,0.031618,0.063277,0.076867,0.085800,0.043602,0.010043,0.010043,0.033267,0.027990
2,0.042498,1.000000,0.171949,0.046380,0.135857,0.017796,0.046014,0.111582,0.038939,0.145693,...,0.118255,0.076802,0.165593,0.124172,0.154723,0.146398,0.026283,0.026283,0.114125,0.073248
3,0.251040,0.171949,1.000000,0.055890,0.072789,0.021445,0.055449,0.066736,0.000000,0.078059,...,0.070727,0.030548,0.061136,0.074266,0.082897,0.475086,0.009704,0.009704,0.032141,0.027043
4,0.057847,0.046380,0.055890,1.000000,0.082219,0.024224,0.062633,0.226534,0.104724,0.088172,...,0.240082,0.034506,0.069057,0.252095,0.093637,0.514347,0.010961,0.010961,0.036305,0.030546
5,0.075338,0.135857,0.072789,0.082219,1.000000,0.031548,0.081571,0.481249,0.000000,0.624335,...,0.104046,0.044939,0.202283,0.109252,0.663032,0.061972,0.032107,0.032107,0.106347,0.089477


### Tìm các cặp template giống nhau nhất

In [40]:
similar_pairs = []

template_ids = templates_for_embedding["template_id"].tolist()

for i in range(len(template_ids)):
    for j in range(i + 1, len(template_ids)):
        tid_i = template_ids[i]
        tid_j = template_ids[j]
        score = similarity_df.loc[tid_i, tid_j]

        similar_pairs.append({
            "template_id_1": tid_i,
            "template_id_2": tid_j,
            "similarity": score,
            "template_1": templates_for_embedding.loc[
                templates_for_embedding["template_id"] == tid_i, "template"
            ].iloc[0],
            "template_2": templates_for_embedding.loc[
                templates_for_embedding["template_id"] == tid_j, "template"
            ].iloc[0],
        })

similar_pairs_df = (
    pd.DataFrame(similar_pairs)
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

similar_pairs_path = RESULTS_DIR / "top_template_similar_pairs.csv"
similar_pairs_df.head(20).to_csv(similar_pairs_path, index=False)

print("Saved:", similar_pairs_path)
similar_pairs_df.head(10)

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\top_template_similar_pairs.csv


,template_id_1,template_id_2,similarity,template_1,template_2
0,12,15,0.765162,081110 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>,081111 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
1,8,15,0.721984,081109 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>,081111 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
2,10,16,0.711038,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,081111 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
3,8,12,0.687580,081109 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>,081110 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
4,5,16,0.663032,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,081111 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
5,5,10,0.624335,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
6,4,17,0.514347,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>,081111 <*> <*> INFO dfs.DataNode$DataXceiver: Received block <*> src: <*> dest: <*> of size 67108864
7,5,8,0.481249,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,081109 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
8,10,14,0.475746,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,081110 <*> 19 INFO dfs.FSNamesystem: BLOCK* ask <*> to delete <*>
9,3,17,0.475086,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>,081111 <*> <*> INFO dfs.DataNode$DataXceiver: Received block <*> src: <*> dest: <*> of size 67108864


In [41]:
from sklearn.cluster import AgglomerativeClustering

# distance = 1 - cosine similarity
distance_matrix = 1 - similarity_matrix

# threshold càng thấp càng strict; HDFS_2k ít template nên 0.65 là tương đối ổn
cluster_model = AgglomerativeClustering(
    metric="precomputed",
    linkage="average",
    distance_threshold=0.65,
    n_clusters=None,
)

cluster_labels = cluster_model.fit_predict(distance_matrix)

template_clusters_df = templates_for_embedding.copy()
template_clusters_df["cluster"] = cluster_labels

template_clusters_path = RESULTS_DIR / "template_clusters_tfidf.csv"
template_clusters_df.to_csv(template_clusters_path, index=False)

print("Saved:", template_clusters_path)
print("Number of clusters:", template_clusters_df["cluster"].nunique())

template_clusters_df.sort_values(["cluster", "count"], ascending=[True, False])

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\template_clusters_tfidf.csv
Number of clusters: 14


,template_id,template,count,cluster
13,14,081110 <*> 19 INFO dfs.FSNamesystem: BLOCK* ask <*> to delete <*>,2,0
19,20,081111 080934 19 INFO dfs.FSNamesystem: BLOCK* ask 10.250.14.38:50010 to replicate blk_-7571492020523929240 to datanode(s) 10.251.122.38:50010,1,0
1,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>,314,1
10,11,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>,224,1
15,16,081111 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,55,2
9,10,081110 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,52,2
4,5,081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>,8,2
3,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>,292,3
16,17,081111 <*> <*> INFO dfs.DataNode$DataXceiver: Received block <*> src: <*> dest: <*> of size 67108864,2,3
6,7,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>,263,4


In [42]:
cluster_summary_rows = []

for cluster_id, group in template_clusters_df.groupby("cluster"):
    top_template = group.sort_values("count", ascending=False).iloc[0]

    cluster_summary_rows.append({
        "cluster": cluster_id,
        "num_templates": len(group),
        "total_count": int(group["count"].sum()),
        "representative_template_id": int(top_template["template_id"]),
        "representative_template": top_template["template"],
    })

cluster_summary_df = (
    pd.DataFrame(cluster_summary_rows)
    .sort_values(["total_count", "num_templates"], ascending=[False, False])
)

cluster_summary_path = RESULTS_DIR / "template_cluster_summary.csv"
cluster_summary_df.to_csv(cluster_summary_path, index=False)

print("Saved:", cluster_summary_path)
cluster_summary_df

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\template_cluster_summary.csv


,cluster,num_templates,total_count,representative_template_id,representative_template
1,1,2,538,2,<*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
10,10,1,311,1,<*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating
3,3,2,294,4,<*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>
12,12,1,292,3,<*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>
4,4,1,263,7,<*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>
2,2,3,115,16,081111 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
5,5,3,80,12,081110 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
11,11,1,80,9,<*> <*> <*> WARN dfs.DataNode$DataXceiver: <*> exception while serving <*> to <*>
9,9,1,20,6,<*> <*> 13 INFO dfs.DataBlockScanner: Verification succeeded for <*>
0,0,2,3,14,081110 <*> 19 INFO dfs.FSNamesystem: BLOCK* ask <*> to delete <*>


### Inject một dòng log lạ và detect new template


In [43]:
before_template_ids = set(final_templates_df["template_id"])
before_num_templates = len(before_template_ids)

weird_log = (
    "081111 112233 999 ERROR dfs.SecurityAuditor: "
    "Quantum checksum mismatch for alien_payload_id=ZX-991 "
    "source=/unknown-zone action=teleport severity=critical"
)

result = miner_p2.add_log_message(weird_log)

after_clusters = get_clusters(miner_p2)
after_templates_df = pd.DataFrame([
    {
        "template_id": cluster.cluster_id,
        "template": cluster_template(cluster),
        "count": cluster.size,
    }
    for cluster in after_clusters
]).sort_values("template_id").reset_index(drop=True)

after_template_ids = set(after_templates_df["template_id"])
new_template_ids = sorted(after_template_ids - before_template_ids)

is_new_template = result["cluster_id"] in new_template_ids

print("Injected weird log:")
print(weird_log)
print()
print("Drain3 result:")
print("cluster_id:", result["cluster_id"])
print("template_mined:", result["template_mined"])
print("change_type:", result["change_type"])
print()
print("Templates before:", before_num_templates)
print("Templates after:", len(after_templates_df))
print("New template IDs:", new_template_ids)
print("Detected new template:", is_new_template)

Injected weird log:
081111 112233 999 ERROR dfs.SecurityAuditor: Quantum checksum mismatch for alien_payload_id=ZX-991 source=/unknown-zone action=teleport severity=critical

Drain3 result:
cluster_id: 22
template_mined: 081111 112233 999 ERROR dfs.SecurityAuditor: Quantum checksum mismatch for alien_payload_id=ZX-991 source=/unknown-zone action=teleport severity=critical
change_type: cluster_created

Templates before: 21
Templates after: 22
New template IDs: [22]
Detected new template: True


In [44]:
new_template_detection_df = pd.DataFrame([{
    "raw_log": weird_log,
    "cluster_id": result["cluster_id"],
    "template_mined": result["template_mined"],
    "change_type": result["change_type"],
    "templates_before": before_num_templates,
    "templates_after": len(after_templates_df),
    "is_new_template": is_new_template,
}])

new_template_detection_path = RESULTS_DIR / "injected_new_template_detection.csv"
new_template_detection_df.to_csv(new_template_detection_path, index=False)

print("Saved:", new_template_detection_path)
new_template_detection_df

Saved: d:\DevopsAndCloud\AIOPS\W1\day-2\results\injected_new_template_detection.csv


,raw_log,cluster_id,template_mined,change_type,templates_before,templates_after,is_new_template
0,081111 112233 999 ERROR dfs.SecurityAuditor: Quantum checksum mismatch for alien_payload_id=ZX-991 source=/unknown-zone action=teleport severity=critical,22,081111 112233 999 ERROR dfs.SecurityAuditor: Quantum checksum mismatch for alien_payload_id=ZX-991 source=/unknown-zone action=teleport severity=critical,cluster_created,21,22,True
